In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress
import os

load_dotenv()
openai_clint = OpenAI(
    api_key=os.getenv('API_KEY'),
    base_url=os.getenv('API_URL'),
)

In [8]:
rec = answers[0]

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [10]:
prompt

'Question:\nIs it okay to join the course late if I just found it now?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nYes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.'

In [14]:
eval_result, usage = llm_structured_retry(
    openai_clint,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model='openai/gpt-oss-20b'
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same key information: it confirms that joining late is allowed and states the requirement to submit the project while submissions are still accepted to obtain a certificate. The wording is slightly different but the meaning is identical.', score='good')

In [19]:
def evaluate_aqa(question, answer_orig, answer_llm, model="openai/gpt-oss-20b"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_clint,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [20]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer retains the essential information: it confirms that joining late is allowed and clarifies that to receive a certificate one must submit a project while submissions are still accepted. This matches the original answer’s meaning.', score='good')

In [21]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [22]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/395 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxtjc4c0fs9awj7rmvekyn7t` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7224, Requested 1066. Please try again in 2.175s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

In [ ]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)